In [1]:
# !pip install pandas qdrant-client pyarrow tqdm ipywidgets

In [2]:
import pandas as pd
import json
import numpy as np
import os

from tqdm import tqdm 
from qdrant_client import QdrantClient
from qdrant_client.http import models

# ==========================================
# (โค้ดส่วนที่เหลือของคุณเหมือนเดิมทุกประการครับ)
# ==========================================

# ==========================================
# 1. ตั้งค่าการเชื่อมต่อ (Config)
# ==========================================
# Qdrant รันอยู่ใน Docker แต่พอร์ตทะลุมาที่ localhost
QDRANT_HOST = 'qdrant_db'  
QDRANT_PORT = 6333
COLLECTION_NAME = "skill_taxonomy"

# Path ไปยังไฟล์ Parquet ของคุณ
PARQUET_PATH = "./output/corpus.parquet"

In [3]:
# ==========================================
# 2. เชื่อมต่อ Qdrant & สร้าง Collection
# ==========================================
print("🔄 กำลังเชื่อมต่อกับ Qdrant (localhost:6333)...")
qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

if qdrant_client.collection_exists(COLLECTION_NAME):
    print(f"⚠️ พบ Collection '{COLLECTION_NAME}' อยู่แล้ว")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    print(f"📦 กำลังสร้าง Collection '{COLLECTION_NAME}' ใหม่...")
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=1024, 
            distance=models.Distance.COSINE
        ),
    )

🔄 กำลังเชื่อมต่อกับ Qdrant (localhost:6333)...
⚠️ พบ Collection 'skill_taxonomy' อยู่แล้ว


In [4]:
# ==========================================
# 3. โหลดไฟล์ Parquet และเตรียมข้อมูล
# ==========================================
if not os.path.exists(PARQUET_PATH):
    print(f"❌ ไม่พบไฟล์ {PARQUET_PATH} กรุณาตรวจสอบ Path ให้ถูกต้อง")
else:
    print(f"📂 กำลังอ่านไฟล์ {PARQUET_PATH}...")
    df = pd.read_parquet(PARQUET_PATH)
    
    print(f"🚀 พบข้อมูลทักษะทั้งหมด {len(df)} รายการ กำลังเตรียม Vector...")
    
    points = []
    # ใช้ tqdm เพื่อแสดง Progress Bar ใน Jupyter
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Preparing Vectors"):
        skill_name = str(row['skill_name'])
        embedding = row['embedding']
        
        # จัดการ Format ของ Vector เผื่อตอนเซฟมันกลายเป็น String หรือ Numpy
        if isinstance(embedding, str):
            vector = json.loads(embedding)
        elif isinstance(embedding, np.ndarray):
            vector = embedding.tolist()
        else:
            vector = embedding
            
        points.append(
            models.PointStruct(
                id=i,
                vector=vector,
                payload={"skill_name": skill_name}
            )
        )
    
    # ==========================================
    # 4. โยนเข้า Qdrant (Upsert in Batches)
    # ==========================================
    print("💾 กำลังบันทึกข้อมูลลง Qdrant Database...")
    
    # แบ่งโยนทีละ 500 ก้อนเพื่อไม่ให้โหลดหนักเกินไป
    batch_size = 500
    for i in tqdm(range(0, len(points), batch_size), desc="Upserting to Qdrant"):
        qdrant_client.upsert(
            collection_name=COLLECTION_NAME, 
            points=points[i : i+batch_size]
        )
    
    # ตรวจสอบความถูกต้อง
    collection_info = qdrant_client.get_collection(COLLECTION_NAME)
    print(f"\n✅ ดึงข้อมูลจาก Parquet เข้า Qdrant สำเร็จ! มี Vector ถูกเก็บไว้ทั้งหมด {collection_info.points_count} รายการ")

📂 กำลังอ่านไฟล์ ./output/corpus.parquet...
🚀 พบข้อมูลทักษะทั้งหมด 721 รายการ กำลังเตรียม Vector...


Preparing Vectors: 100%|██████████| 721/721 [00:00<00:00, 3808.96it/s]


💾 กำลังบันทึกข้อมูลลง Qdrant Database...


Upserting to Qdrant: 100%|██████████| 2/2 [00:00<00:00,  2.10it/s]


✅ ดึงข้อมูลจาก Parquet เข้า Qdrant สำเร็จ! มี Vector ถูกเก็บไว้ทั้งหมด 721 รายการ
